# M03C: Prompt Templates & Management

Hardcoding prompts becomes messy fast. Templates let you organize, version, and test prompts like real code.

**Topics:**
- Template system with variable substitution
- Prompt library with versioning
- Version comparison workflow

---

## 🔧 Step 1: Setup

In [ ]:
import os
import re
from pathlib import Path
from dotenv import load_dotenv

import openai

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"


def ask_openai(prompt, model=MODEL):
    """Send a prompt to OpenAI and return the response."""
    try:
        response = client.responses.create(
            model=model, 
            input=prompt
        )
        return response.output_text.strip()
            
    except openai.AuthenticationError:
        return "Error: Invalid API key. Check your .env file."
    except openai.RateLimitError:
        return "Error: Rate limit exceeded. Wait a moment and try again."
    except openai.APIConnectionError:
        return "Error: Network connection issue. Check your internet."
    except openai.BadRequestError:
        return "Error: Bad request. Check model name."
    except Exception as e:
        return f"API Error: {str(e)}"

def _escape_braces(text):
    """Escape curly braces so .format() doesn't treat them as placeholders."""
    return str(text).replace("{", "{{").replace("}", "}}")


print("✅ Ready!")

**Note:** This notebook uses a simple `ask_openai` that returns a plain string. This keeps the focus on templates instead of error handling. 

For production, pair these templates with `ask_openai_v3` from M03B — just adjust for its dict return format.

---

## 🎯 The Problem: Hardcoded Prompts

In [ ]:
# ❌ BAD APPROACH: Hardcoded prompts scattered throughout code

def classify_email_bad(email):
    prompt = f"Classify this email as sales, technical, or billing: {email}"
    return ask_openai(prompt)

def summarize_email_bad(email):
    prompt = f"Summarize this email in one sentence: {email}"
    return ask_openai(prompt)

def extract_action_items_bad(email):
    prompt = f"Extract action items from this email: {email}"
    return ask_openai(prompt)


# --------------------------------------------------------------
# Demonstrate the problem
# --------------------------------------------------------------
print("❌ HARDCODED PROMPTS DEMO")
print("="*60)

test_email = "Can you send me a quote for 50 licenses?"
print(f"Email: {test_email}\n")

print("Classification: ", end="")
print(classify_email_bad(test_email))

print("\nSummary: ", end="")
print(summarize_email_bad(test_email))

print("\n" + "="*60)

### ❌ Problems with This Approach

- **Prompts scattered** across codebase
- **Hard to update** — must find and edit each one
- **Hard to compare** — no clean way to test different versions

---

## 💡 Solution 1: Basic Templates

Extract prompts into reusable templates with variables:

In [ ]:
# ✅ BETTER APPROACH: Templates with variables

TEMPLATES = {
    "email_classifier": """Classify this customer email into one category: 
{categories}.

Email: {email}

Return only the category name.""",
    
    "email_summarizer": """Summarize this email in {length}.

Email: {email}

Summary:""",
    
    "action_extractor": """Extract action items from this email.

Email: {email}

List each action item on a new line starting with '-'."""
}

def render_template(template_name, **variables):
    if template_name not in TEMPLATES:
        raise ValueError(f"Template '{template_name}' not found")
    
    template = TEMPLATES[template_name]
    
    # Escape any curly braces in variable values
    escaped_variables = {k: _escape_braces(v) for k, v in variables.items()}
    
    return template.format(**escaped_variables)


# --------------------------------------------------------------
# Test the template system
# --------------------------------------------------------------
print("✅ TEMPLATE SYSTEM")
print("="*60)

email = "Can you send me a quote for 50 licenses?"

prompt = render_template(
    "email_classifier",
    categories="sales, technical, billing, general",
    email=email
)

print(f"Email: {email}")
print(f"\nGenerated prompt:\n{prompt}")
print(f"\nClassification: ", end="")
result = ask_openai(prompt)
print(result)

print("\n" + "="*60)

### ✅ Benefits of Templates

- **All prompts in one place** — easy to find and update
- **Reusable with variables** — one template, many uses
- **Consistent formatting** — all prompts follow the same structure

---

## 🏗️ Solution 2: Template Classes

Create a proper class for better organization, validation, and features:

In [ ]:
class PromptTemplate:
    def __init__(self, name, template, version="1.0", description=""):
        self.name = name
        self.template = template
        self.version = version
        self.description = description
    
    def render(self, **variables):
        try:
            # Escape any curly braces in variable values
            escaped_variables = {k: _escape_braces(v) for k, v in variables.items()}
            return self.template.format(**escaped_variables)
        except KeyError as e:
            raise ValueError(f"Missing variable: {e}")
    
    def get_variables(self):
        return re.findall(r'\{(\w+)\}', self.template)
    
    def __repr__(self):
        return f"PromptTemplate('{self.name}', v{self.version})"


# --------------------------------------------------------------
print("✅ PromptTemplate class defined!")

### Using the Template Class

Now that we've defined the tool, let's use it to build a robust classifier:

In [ ]:
email_classifier_v1 = PromptTemplate(
    name="email_classifier",
    template="""Classify this email into one category: 
{categories}.

Email: {email}

Return only the category name.""",
    version="1.0",
    description="Classifies customer emails by department"
)


# --------------------------------------------------------------
# Use the template
# --------------------------------------------------------------
print("🏗️ TEMPLATE CLASS USAGE")
print("="*60)
print(f"Template: {email_classifier_v1}")
print(f"Description: {email_classifier_v1.description}")
print(f"Variables: {email_classifier_v1.get_variables()}")

prompt = email_classifier_v1.render(
    categories="sales, technical, billing",
    email="The app crashed when I clicked export."
)

print(f"\nResult: ", end="")
result = ask_openai(prompt)
print(result)

print("\n" + "="*60)

### ✅ Class Benefits

- **Version tracking** — know which template (name + version) produced results
- **Safer rendering** — missing variables raise clear errors before API call
- **Self-documenting** — name, version, and description built in

---

## 📚 Solution 3: Prompt Library

Centralize all PromptTemplates in a library for easy management:

In [ ]:
class PromptLibrary:
    """Central repository for all PromptTemplates."""
    
    def __init__(self):
        self.templates = {}  # {name: {version: template}}
    
    def register(self, template):
        """Add a PromptTemplate to the library."""
        if template.name not in self.templates:
            self.templates[template.name] = {}
        
        self.templates[template.name][template.version] = template
        print(f"✅ Registered: {template.name} v{template.version}")
    
    def get(self, name, version="latest"):
        """Retrieve a template by name and version."""
        if name not in self.templates:
            raise ValueError(f"Template '{name}' not found")
        
        versions = self.templates[name]
        
        if version == "latest":
            def version_key(v):
                try:
                    return tuple(map(int, v.split(".")))
                except ValueError:
                    return (0,)
            latest_version = max(versions.keys(), key=version_key)
            return versions[latest_version]
        
        if version not in versions:
            raise ValueError(f"Version '{version}' not found for '{name}'")
        
        return versions[version]
    
    def list_templates(self):
        """List all available templates."""
        result = []
        for name, versions in self.templates.items():
            for version, template in versions.items():
                result.append(f"{name} (v{version})")
        return result
    
    def list_versions(self, template_name):
        """List all versions of a template."""
        if template_name not in self.templates:
            return []
        return list(self.templates[template_name].keys())


# --------------------------------------------------------------
print("✅ PromptLibrary class defined!")

### 🗂️ How the Library Stores Templates

Inside the library, templates are stored in a nested dictionary:
```python
self.templates = {
    "sentiment": {
        "1.0": PromptTemplate(...),
        "2.0": PromptTemplate(...),
    },
    "classifier": {
        "1.0": PromptTemplate(...),
    },
}
```

- **First key:** Template name
- **Second key:** Version string
- **Value:** The `PromptTemplate` object

So `library.get("sentiment", version="2.0")` returns `self.templates["sentiment"]["2.0"]`.

### 🔧 Using the Library

Let's put the library to work by registering and using different templates:

In [ ]:
library = PromptLibrary()

library.register(PromptTemplate(
    name="classifier",
    template="""Classify: {text}
Categories: {categories}
Category:""",
    version="1.0",
    description="Simple classifier"
))

library.register(PromptTemplate(
    name="summarizer",
    template="""Summarize in {length}:
{text}
Summary:""",
    version="1.0",
    description="Text summarizer"
))


# --------------------------------------------------------------
# Use the library
# --------------------------------------------------------------
print("📚 PROMPT LIBRARY USAGE")
print("="*60)

print("Available templates:")
for template in library.list_templates():
    print(f"  - {template}")

template = library.get("classifier", version="latest")
prompt = template.render(
    text="Great product!",
    categories="positive, negative, neutral"
)

print(f"\nUsing template: {template}")
print(f"Result: ", end="")
result = ask_openai(prompt)
print(result)

print("\n" + "="*60)

### ✅ Library Benefits

- **Central repository** — all templates in one place
- **Version management** — track and retrieve specific versions
- **"Latest" keyword** — automatically get newest version

### 💡 Note on These Classes

You don't need to memorize this implementation. Treat `PromptTemplate` and `PromptLibrary` as **utilities you can copy into your own projects** and customize as needed.

The important thing is understanding *why* we structure prompts this way — not memorizing every line of code.

---

## 🔄 Solution 4: Version Control

Track changes and compare versions:

In [ ]:
# New library for the versioning demo
library = PromptLibrary()

# Version 1.0 - Basic
library.register(PromptTemplate(
    name="sentiment",
    template="What is the sentiment of: {text}",
    version="1.0",
    description="Basic sentiment analysis"
))

# Version 2.0 - Explicit instructions
library.register(PromptTemplate(
    name="sentiment",
    template="""Analyze the sentiment of this text.

Text: {text}

Return one word: positive, negative, or neutral.""",
    version="2.0",
    description="Improved with explicit format"
))

# Version 3.0 - Role and examples
library.register(PromptTemplate(
    name="sentiment",
    template="""You are a sentiment analysis expert.

Examples:
Text: "Love it!"
positive

Text: "Terrible quality."
negative

Now analyze:
Text: {text}""",
    version="3.0",
    description="Added role and few-shot examples"
))


# --------------------------------------------------------------
# Test all versions
# --------------------------------------------------------------
print("🔄 VERSION CONTROL")
print("="*60)

test_text = "This product is okay, nothing special."
print(f"Test input: {test_text}")
print("\nResults by version:")

for version in library.list_versions("sentiment"):
    template = library.get("sentiment", version=version)
    prompt = template.render(text=test_text)
    result = ask_openai(prompt)
    print(f"  v{version}: {result}")

print("="*60)

### ✅ Version Control Benefits

- **Track what changed** — descriptions document improvements
- **Easy rollback** — revert to any previous version
- **Safe experimentation** — keep working version while testing new

---

## 📋 Best Practices Summary

### 1. Use Templates
```python
# ❌ Bad: Hardcoded
prompt = f"Classify: {text}"

# ✅ Good: Template
prompt = template.render(text=text)
```

### 2. Centralize Prompts
```python
# ✅ All prompts in one place
library = PromptLibrary()
template = library.get("classifier", version="latest")
```

### 3. Version Everything
```python
# Register multiple versions to the library
library.register(PromptTemplate(..., version="1.0"))
library.register(PromptTemplate(..., version="2.0"))

# Easy rollback if v2 performs worse
template = library.get("classifier", version="1.0")
```

### 4. Document Changes
```python
PromptTemplate(
    name="classifier",
    template="...",
    version="2.0",
    description="Added role and explicit instructions"  # ✅
)
```

### 5. Test Variations
```python
# ✅ Compare versions before deploying
for version in library.list_versions("classifier"):
    template = library.get("classifier", version=version)
    result = ask_openai(template.render(text=test_input))
    print(f"v{version}: {result}")
```

---

### 💪 Your Turn: Build Your Own System

Create a template library for a Support Ticket Classifier.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise: Build Your Own Template System
# --------------------------------------------------------------
# Objective: Create a template library for a "Support Ticket Classifier" and compare 2 versions.

# 1. Setup
my_library = PromptLibrary()

# TODO 1: Create Version 1 Template (Simple)
# Categories: technical, billing, sales, general
# Use {ticket} as the placeholder for the support ticket text
# t1 = PromptTemplate(name="ticket_classifier", version="1.0", template="...")
# my_library.register(t1)

# TODO 2: Create Version 2 Template (Better - e.g., with Examples)
# t2 = PromptTemplate(name="ticket_classifier", version="2.0", template="...")
# my_library.register(t2)

# TODO 3: Define Test Cases
# test_tickets = [
#     "I can't log in",
#     "How much is the pro plan?",
#     "The export button doesn't work"
# ]

# TODO 4: Compare versions
# for ticket in test_tickets:
#     print(f"Ticket: {ticket}")
#     for version in my_library.list_versions("ticket_classifier"):
#         template = my_library.get("ticket_classifier", version=version)
#         result = ask_openai(template.render(ticket=ticket))
#         print(f"  v{version}: {result}")
#     print()

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

### What You Learned

**Prompt Templates:**
- Extract prompts from code into reusable templates
- Use variables for dynamic content
- Separate prompt logic from application logic

**Version Control:**
- Track prompt versions like code
- Document what changed and why
- Easy rollback if new version performs worse

**Prompt Libraries:**
- Centralize all prompts in one place
- Organize by name and version
- Easy discovery of existing prompts

**Comparing Versions:**
- Test prompt variations side-by-side
- Identify which version performs better
- Scale with automated tools in production

### Quick Reference

**The Flow:** Template → Library → Version → Compare → Deploy winner

---

### 📍 Next Step

**M03D: Prompt Evaluation** — Build test sets and evaluation harnesses to measure prompt performance.

---

## 🔧 Troubleshooting

**Template rendering fails?**
- Check all variables are provided
- Use `.get_variables()` to see required variables
- Verify variable names match exactly (case-sensitive)

**Can't find template version?**
- Use `.list_versions(name)` to see available versions
- Check version string format matches exactly
- Try "latest" to get most recent version

**Versions producing similar results?**
- Make bigger changes between versions
- Test with diverse, real-world examples
- Try completely different approaches

**How to organize large libraries?**
- Group templates by feature/domain
- Use naming conventions (feature_task_v1)
- Add descriptions to each template

**Variable errors?**
- Use `.get_variables()` before rendering
- Provide default values when possible
- Validate input before rendering

---